In [4]:
import math
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import scipy
from scipy import stats

# Having a conversation with our data
### When should BI tool or assisant ask a clarifying question? A Bayes-Shannon model tested on AdventureWorks

**Author:** baxendy (AG)

**Abstract.** *Business-intelligence (BI) tools now let people ask questions about their data in plain language. These questions are often ambiguous: 'show sales by region' does not say whether 'sales' means money or units, what level of geography is needed, or which period. Most assistants still answer with full confidence, so a wrong number looks as believable as a correct one. This notebook builds a small mathematical model of the conversation between a user and a BI assistant. The assistant counts the possible interpretations of a request (combinatorics), estimates their probabilities with Bayes' theorem (a Naive Bayes classifier), measures its uncertainty with Shannon entropy and asks a clarifying question while the entropy is above a threshold. The model is tested with 600 simulated requests to Microsoft's AdventureWorks sample data warehouse, with and without a semantic layer, with and without clarifying questions.*

**Contents**
1. The problem and the hypotheses
2. Mathematical toolbox

## 1. The problem and the hypotheses

BI platforms let users ask questions in natural language. A request like *show sales by region* leaves three things open on a typical sales model:

* the **measure**: sales amount (money) or order quantity (units);
* the **grouping**: sales-territory region, country or territory group;
* the **period**: last year, the year before, or all years.

A human analyst would ask back, but an assistant usually picks one interpretation and answers confidently.

The semantic layer (measures, synonyms, descriptions, default rules) helps the assistant to understand requests (Sequeda et al., 2023), and clarifying questions help to fix what is still unclear (Zhao et al., 2024). The idea that a system should decide between acting and asking is called *mixed initiative* (Horvitz, 1999). This project tests one simple, measurable rule for that decision:

> **Ask while the uncertainty of the assistant (Shannon entropy) is above a threshold τ; otherwise answer.**

| | Hypothesis |
|---|---|
| **H1** | A semantic layer lowers the initial uncertainty of the assistant. |
| **H2** | Higher uncertainty goes together with wrong answers. |
| **H3** | Clarifying questions increase the accuracy of the answers. |
| **H4** | With a semantic layer fewer clarifying questions are needed. |

**Assumptions.** The users are simulated. A request has three or four independent parts. The simulated user answers questions correctly, but a user who makes mistakes is also tested.



## 2. Mathematical toolbox

### 2.1 Counting interpretations: the rule of product

An interpretation chooses one value for every part of a request. If part $i$ has $n_i$ possible values, the rule of product gives the number of interpretations

$$N = n_1 \cdot n_2 \cdots n_k = \prod_{i=1}^{k} n_i .$$

For *show sales by region* with 2 measures, 3 groupings and 3 periods this is already $2 \cdot 3 \cdot 3 = 18$. A grouping is a dimension attribute that the data can be grouped by: here region, country and territory group, three levels of the same sales-territory dimension. Our semantic model has 6 measures, 6 groupings (from the territory, product and date dimensions) and 4 periods. Without a semantic layer the assistant only sees raw database columns, so it must also guess the aggregation (SUM, AVERAGE or COUNT): the measure splits into a column and an aggregation.


In [19]:
n_small_example = 2 * 3 * 3
n_with_layer = 6 * 6 * 4
n_without_layer = 6 * 3 * 6 * 4

print("Interpretations of 'show sales by region' :", n_small_example)
print("Interpretations with a semantic layer:" , n_with_layer)
print("Interpretations without a semantic layer:" , n_without_layer)

Interpretations of 'show sales by region' : 18
Interpretations with a semantic layer: 144
Interpretations without a semantic layer: 432


### 2.2 Bayes' theorem

Before reading a request $q$ the assistant has a prior probability $P(t)$ for every interpretation $t$. After reading it, Bayes' theorem gives the posterior probability

$$P(t \mid q) = \frac{P(q \mid t)\,P(t)}{P(q)}, \qquad P(q) = \sum_{t'} P(q \mid t')\,P(t').$$

$P(q \mid t)$, the likelihood, is how probable it is that a user who wants $t$ writes $q$. The denominator $P(q)$ comes from the law of total probability; it makes the posteriors sum to 1.

*Example.* The word 'sales' is used for the sales amount with probability 0.30 and for the order quantity with probability 0.20, and both measures are equally likely beforehand. A semantic layer that defines 'sales' as a synonym of Sales Amount changes the likelihoods to 0.38 and 0.02.


In [20]:
def bayes(likelihoods, priors):

    prob = [likelihood * prior for likelihood, prior in zip(likelihoods, priors)]
    total = sum(prob)
    
    return [round(value / total, 3) for value in prob]


print("'sales' without a semantic layer:", bayes([0.30, 0.20], [0.5, 0.5]))
print("'sales' with a semantic layer:", bayes([0.38, 0.02], [0.5, 0.5]))

'sales' without a semantic layer: [0.6, 0.4]
'sales' with a semantic layer: [0.95, 0.05]


### 2.3 Naive Bayes

A request has several words $w_1, w_2, \dots$ Naive Bayes assumes that the words are independent when the meaning is known, so

$$P(q \mid t) = \prod_{j} P(w_j \mid t).$$

The word probabilities are learned by counting: how often does the word $w$ appear in the examples of class $c$? With Laplace smoothing ($\alpha = 1$ is added to every count) a word that was never seen with a class does not get probability 0:

$$P(w \mid c) = \frac{\text{count}(w, c) + \alpha}{\text{count}(c) + \alpha V},$$

where $\text{count}(c)$ is the number of words seen with class $c$ and $V$ is the number of different words. Words that never appeared in the training examples are skipped. At the end every product is divided by the sum over all classes, exactly as in Bayes' theorem above, so the probabilities sum to 1.


In [ ]:
STOP_WORDS = ['a', 'an', 'and', 'are', 'break', 'by', 'can', 'did', 'do', 'down',
              'each', 'for', 'give', 'have', 'how', 'in', 'is', 'list', 'me', 'much',
              'of', 'on', 'per', 'please', 'show', 'since', 'the', 'this', 'to',
              'was', 'we', 'what', 'you']


def tokenize(text):

    clean = ''

    for character in text.lower():
        if 'a' <= character <= 'z':
            clean = clean + character
        else:
            clean = clean + ' '
            
    return [word for word in clean.split() if word not in STOP_WORDS]


print(tokenize('Show total revenue by Country, last year!'))

['total', 'revenue', 'country', 'last', 'year']


In [ ]:
def train_naive_bayes(examples, classes, prior=None):

    counts = {}
    totals = {}
    vocabulary = set()

    for c in classes:
        counts[c] = {}
        totals[c] = 0

    for text, label in examples:
        for word in tokenize(text):
            counts[label][word] = counts[label].get(word, 0) + 1
            totals[label] = totals[label] + 1
            vocabulary.add(word)

    if prior is None:
        prior = [1 / len(classes)] * len(classes)

    return {'classes': classes, 'counts': counts, 'totals': totals, 'vocabulary': vocabulary, 'prior': prior}


def word_probability(model, word, c):

    V = len(model['vocabulary'])
    return (model['counts'][c].get(word, 0) + 1) / (model['totals'][c] + V)


def predict(model, text):

    prob = []

    for c, prior in zip(model['classes'], model['prior']):
        p = prior
        for word in tokenize(text):
            if word in model['vocabulary']:
                p = p * word_probability(model, word, c)
        prob.append(p)

    total = sum(prob)

    return np.array([value / total for value in prob])


toy_model = train_naive_bayes(
    [('sales amount', 'Sales Amount'), ('revenue', 'Sales Amount'), ('sales', 'Sales Amount'),
     ('order quantity', 'Order Quantity'), ('units sold', 'Order Quantity'), ('sales units', 'Order Quantity')],
    ['Sales Amount', 'Order Quantity'])

print('P(class | 'total sales by region') =', predict(toy_model, 'total sales by region').round(3))

P(class | 'total sales by region') = [0.639 0.361]
